### Data Ingestion


In [1]:
### document structure

from langchain_core.documents import Document

c:\Users\mitta\OneDrive\Desktop\emerging lab\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [3]:
doc=Document(
    page_content="This is the main content i am using to create RAG",
    metadata={
        "source":"example.txt",
        "pages":"1",
        "author":"Anshikka",
        "date created": "28-03-2026"
    }
)
doc

Document(metadata={'source': 'example.txt', 'pages': '1', 'author': 'Anshikka', 'date created': '28-03-2026'}, page_content='This is the main content i am using to create RAG')

In [4]:
## Create a simple txt file
import os
os.makedirs("../data/text_files",exist_ok=True)

In [5]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """

}

for filepath,content in sample_texts.items():
    with open(filepath,'w',encoding="utf-8") as f:
        f.write(content)

print("✅ Sample text files created!")

✅ Sample text files created!


In [7]:
### TextLoader

from langchain_community.document_loaders import TextLoader

loader=TextLoader("../data/text_files/python_intro.txt",encoding="utf-8")
document=loader.load()
print(document)

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


In [8]:
### Directory Loader
from langchain_community.document_loaders import DirectoryLoader

## load all the text files from the directory
dir_loader=DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt", ## Pattern to match files  
    loader_cls= TextLoader, ##loader class to use
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=False

)

documents=dir_loader.load()
documents

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems\n\n\n    '),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popu

In [9]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

## load all the text files from the directory
dir_loader=DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf", ## Pattern to match files  
    loader_cls= PyMuPDFLoader, ##loader class to use
    show_progress=False

)

pdf_documents=dir_loader.load()
pdf_documents

[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'EXCEL.EXE', 'creationdate': '', 'source': '..\\data\\pdf\\Club details 2026.pdf', 'file_path': '..\\data\\pdf\\Club details 2026.pdf', 'total_pages': 12, 'format': 'PDF 1.7', 'title': 'b7304b04d805f29b4c305d14648da60793abd00f52757882d41cabd851b9c585.xlsx', 'author': 'Work5', 'subject': '', 'keywords': '', 'moddate': '2026-03-27T21:41:35+00:00', 'trapped': '', 'modDate': 'D:20260327214135Z', 'creationDate': '', 'page': 0}, page_content='S.No\nChapter/Club\nDepartment\nDate, \nMonth & \nYear of \nestablishe\nmnt of \nClub/Cha\npter\nType of \nClub  \n(Chapter/\nDepartme\nntal/Functi\nonal)\nNature of \nClub\n1\n180 Degrees Consulting MUJ Club\nPLACEMENT CELL\nAugust - \n2019\nChapter\nStudent \nChapter\n2\nAbhigya Club\nDepartment of ECE, \nSEEC\n1 Aug \n2018\nFunctional \nClub\nNon-Tech\n3\nACM MUJ STUDENT CHAPTER\nDepartment of IT\nDecember \n14, 2015\nDepartmen\ntal\nStudent \nChapter\n4\nGoonj- The Radio Society\nDept of

In [10]:
type(pdf_documents[0])

langchain_core.documents.base.Document

### embedding and vectorstoreDB


In [12]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [13]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 1 PDF files to process

Processing: Club details 2026.pdf
  ✓ Loaded 12 pages

Total documents loaded: 12


In [14]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [15]:
chunks=split_documents(all_pdf_documents)
chunks

Split 12 documents into 24 chunks

Example chunk:
Content: S.No Chapter/Club Department 
Date, 
Month & 
Year of 
establishe 
mnt of 
Club/Cha 
pter 
Type of 
Club  
(Chapter/ 
Departme 
ntal/Functi 
onal) 
Nature of 
Club 
1 180 Degrees Consulting MUJ Club P...
Metadata: {'producer': 'www.ilovepdf.com', 'creator': 'EXCEL.EXE', 'creationdate': '', 'title': 'b7304b04d805f29b4c305d14648da60793abd00f52757882d41cabd851b9c585.xlsx', 'author': 'Work5', 'moddate': '2026-03-27T21:41:35+00:00', 'source': '..\\data\\pdf\\Club details 2026.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'source_file': 'Club details 2026.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'EXCEL.EXE', 'creationdate': '', 'title': 'b7304b04d805f29b4c305d14648da60793abd00f52757882d41cabd851b9c585.xlsx', 'author': 'Work5', 'moddate': '2026-03-27T21:41:35+00:00', 'source': '..\\data\\pdf\\Club details 2026.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'source_file': 'Club details 2026.pdf', 'file_type': 'pdf'}, page_content='S.No Chapter/Club Department \nDate, \nMonth & \nYear of \nestablishe \nmnt of \nClub/Cha \npter \nType of \nClub  \n(Chapter/ \nDepartme \nntal/Functi \nonal) \nNature of \nClub \n1 180 Degrees Consulting MUJ Club PLACEMENT CELL August - \n2019 Chapter Student \nChapter \n2 Abhigya Club Department of ECE, \nSEEC \n1 Aug \n2018 \nFunctional \nClub Non-Tech \n3 ACM MUJ STUDENT CHAPTER Department of IT December \n14, 2015 \nDepartmen \ntal \nStudent \nChapter \n4 Goonj- The Radio Society Dept of JMC 4-Feb-20 Departmen \ntal Cultural \n5 Nexus (Satellite Club MUJ) Dept of CSE December 

In [16]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\mitta\OneDrive\Desktop\emerging lab\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


c:\Users\mitta\OneDrive\Desktop\emerging lab\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mitta\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4259.69it/s]
BertModel 

Model loaded successfully. Embedding dimension: 384


### VectorStore

In [18]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [19]:
chunks

[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'EXCEL.EXE', 'creationdate': '', 'title': 'b7304b04d805f29b4c305d14648da60793abd00f52757882d41cabd851b9c585.xlsx', 'author': 'Work5', 'moddate': '2026-03-27T21:41:35+00:00', 'source': '..\\data\\pdf\\Club details 2026.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'source_file': 'Club details 2026.pdf', 'file_type': 'pdf'}, page_content='S.No Chapter/Club Department \nDate, \nMonth & \nYear of \nestablishe \nmnt of \nClub/Cha \npter \nType of \nClub  \n(Chapter/ \nDepartme \nntal/Functi \nonal) \nNature of \nClub \n1 180 Degrees Consulting MUJ Club PLACEMENT CELL August - \n2019 Chapter Student \nChapter \n2 Abhigya Club Department of ECE, \nSEEC \n1 Aug \n2018 \nFunctional \nClub Non-Tech \n3 ACM MUJ STUDENT CHAPTER Department of IT December \n14, 2015 \nDepartmen \ntal \nStudent \nChapter \n4 Goonj- The Radio Society Dept of JMC 4-Feb-20 Departmen \ntal Cultural \n5 Nexus (Satellite Club MUJ) Dept of CSE December 

In [20]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 24 texts...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.55s/it]

Generated embeddings with shape: (24, 384)
Adding 24 documents to vector store...
Successfully added 24 documents to vector store
Total documents in collection: 24


### Retrieval Pipeline

In [ ]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 8, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [22]:
rag_retriever

In [24]:
rag_retriever.retrieve("When was IEEE established")

Retrieving documents for query: 'When was IEEE established'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 57.48it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)


[]